# 04b · Fondo local (superficie)

**Spec:** —  |  **Bloque:** C · Extracción  |  **Run de este set:** `ROXs12b_realigned`

Sustrae una superficie local al fondo alrededor del compañero.

| | |
|---|---|
| **Entrada** | Cubo alineado (native_stage02) |
| **Salida (QC/productos)** | `stages/stage04b_qc.json` |
| **Consume aguas abajo** | C2/C3 (control local) |


## Qué hace 04b y por qué

04b resta una **superficie local** (un **plano**) ajustada al fondo en un **anillo** alrededor del compañero (máscara 3 px en el núcleo, ajuste hasta 12 px), **canal por canal**. Así elimina el **gradiente del halo estelar** en la posición del compañero → un fondo local limpio para extraer su espectro sin el pedestal del halo.

Trabaja en modo `native_stage02`: directamente sobre el cubo alineado de B2 (la rama de **sustracción local**, no la de PCA/Stage04). Ajusta el plano con **sigma-clip** (3σ, 3 iteraciones, mínimo 30 px) enmascarando el núcleo del compañero (3 px) y otros objetos (la estrella) dentro de 3 px.

También calcula la **máscara de longitud de onda** buena/mala (ventana del láser AO 5780–6050 Å, 216 canales) y marca los canales de Hα/Hβ y las ventanas de continuo para aguas abajo.

**Rol en la cadena:** produce `cube_residual_local_object.fits`, el cubo del objeto con el fondo local restado; es la base de la rama de extracción **local-surface** (`optimal_ls`) y de la cadena de objetos lejanos (06/07/08). *(Ojo: D1 documenta que `optimal_ls` arrastraba este pedestal — por eso el canónico es psffit, no optimal_ls.)*


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs12b_realigned   # el run de este objeto
cd MUSE-accretion-pipeline                    # raíz del repo
python -m musepipe.stages.stage04b_local_surface --run-id $RUN
```

Ligero–moderado.

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
RUN_ID = nb.resolve_run_id('ROXs12b_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/stage04b_qc.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'python -m musepipe.stages.stage04b_local_surface --run-id $RUN'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/stage04b_qc.json', RUN_ID)
nb.show(qc, keys=['method', 'local_model_kind', 'fit_radius_px', 'target_yx', 'bad_channel_count'], title='04b')


## Resultados que llevaron a la conclusión

Parámetros del ajuste local y productos del `stage04b_qc.json`.


In [ ]:
if qc is None:
    print('(evidencia omitida: la etapa no se ha ejecutado para esta cadena)')
else:
    with nb.evidence_guard('04b', 'stages/stage04b_qc.json'):
        q = nb.load_qc('stages/stage04b_qc.json', RUN_ID)
        print('método:', q['method'], '| input_mode:', q['input_mode'], '| modelo:', q['local_model_kind'])
        print(f"objetivo: '{q['target_object']}' en (y,x)={q['target_yx']}")
        print(f"ajuste: anillo máscara {q['mask_radius_px']:.0f}px .. fit {q['fit_radius_px']:.0f}px; "
              f"sigma-clip {q['local_fit_sigma_clip']} x{q['local_fit_max_iter']}, min {q['local_fit_min_pixels']}px; "
              f"nfit mediana {q['nfit_median_per_cube'][0]:.0f}px")
        print(f"máscara otros objetos: {q['mask_other_objects']} (r={q['other_mask_radius_px']:.0f}px)")
        print(f"canales malos (láser AO {q['bad_wavelength_ranges_A']}): {q['bad_channel_count']}")
        print(f"Hα en {q['halpha_channels_A']} Å  |  Hβ en {q['hbeta_channels_A']} Å")
        print(f"salida: {q['object_cube_fits'].split('/')[-1]}  (elapsed {q['elapsed_s']:.1f}s)")


## Plot — fondo local antes/después (stamp del compañero)

**FITS usados:** el cubo de entrada (`stage02_xcorr_cube_stack.fits`) y el de salida (`cube_residual_local_object.fits`), colapsados en la banda de continuo de Hα (6570–6700 Å) en un recorte alrededor del compañero. **Izq:** el gradiente del halo estelar. **Der:** dentro del anillo de ajuste (cyan, 12 px) el fondo queda **plano** (plano local restado); el núcleo del compañero (rojo, 3 px) se enmascara del ajuste.


In [ ]:
try:
    MAKE_PLOT = True   # dos cubos (~0.4 GB c/u); requiere kernel MUSE
    if MAKE_PLOT:
        try:
            import numpy as np
            import matplotlib.pyplot as plt
            from matplotlib.patches import Circle
            from astropy.io import fits
            from musepipe.io import read_wavelength_axis
            from musepipe.stages.stage04b_local_surface import CONT_HA_RANGE_A

            q = nb.load_qc('stages/stage04b_qc.json', RUN_ID)
            ty, tx = q['target_yx']; fr = q['fit_radius_px']; mr = q['mask_radius_px']
            with fits.open(q['input_cube_fits'], memmap=True) as _h:
                wave = read_wavelength_axis(_h)   # ext WAVELENGTH del stack
            sel = (wave >= CONT_HA_RANGE_A[0]) & (wave <= CONT_HA_RANGE_A[1])   # continuo Hα oficial
            def band_img(path):
                h = fits.open(path, memmap=True)
                hd = next(x for x in h if x.data is not None and np.asarray(x.data).ndim >= 3)
                d = np.asarray(hd.data, dtype=np.float32); d = d[0] if d.ndim == 4 else d
                img = np.nanmedian(d[sel], axis=0); h.close(); return img
            S = 22
            imgi = band_img(q['input_cube_fits']); imgo = band_img(q['object_cube_fits'])
            sub = lambda im: im[ty - S:ty + S, tx - S:tx + S]
            ci, co = sub(imgi), sub(imgo)
            v = np.nanpercentile(ci, [5, 99])
            fig, axes = plt.subplots(1, 2, figsize=(11, 5))
            for ax, im, title in [(axes[0], ci, 'entrada (stage02): gradiente del halo'),
                                  (axes[1], co, 'salida 04b: fondo local restado')]:
                ax.imshow(im, origin='lower', cmap='viridis', vmin=v[0], vmax=v[1])
                ax.add_patch(Circle((S, S), mr, fill=False, ec='red', lw=1.5))
                ax.add_patch(Circle((S, S), fr, fill=False, ec='cyan', lw=1.5, ls='--'))
                ax.set_title(title); ax.axis('off')
            axes[0].plot([], [], color='red', label=f'máscara {mr:.0f}px')
            axes[0].plot([], [], color='cyan', ls='--', label=f'anillo ajuste {fr:.0f}px')
            axes[0].legend(fontsize=8, loc='upper right')
            fig.suptitle(f'04b · plano local (anillo {mr:.0f}-{fr:.0f}px) alrededor del compañero · continuo Hα')
            fig.tight_layout()
            outdir = nb.run_dir(RUN_ID) / 'plots' / 'c_04b'; outdir.mkdir(parents=True, exist_ok=True)
            fig.savefig(outdir / 'local_surface.png', dpi=110)
            print('figura ->', outdir / 'local_surface.png'); plt.show()
        except Exception as e:
            print('No se pudo generar el plot:', type(e).__name__, e)
except FileNotFoundError as e:
    print('[etapa pendiente para este objeto]', e)


## Decisiones y notas
- **Sustracción de plano local en anillo (3–12 px)** alrededor del compañero, canal por canal, en modo `native_stage02`: quita el gradiente del halo estelar localmente. · [`docs/roxs12b_clean_spectrum_pipeline.md`](../docs/roxs12b_clean_spectrum_pipeline.md)
- Enmascara el núcleo del compañero (3 px) y otros objetos (estrella); plano con sigma-clip 3σ.
- Es la rama **local-surface** (`optimal_ls` / cadena de objetos lejanos); D1 mostró que `optimal_ls` arrastraba este pedestal → el canónico es **psffit**, no optimal_ls.


## Conclusión (registrada)

**04b: plano local restado en un anillo 3–12 px alrededor del compañero (156, 76); salida `cube_residual_local_object.fits`.**

- **Fecha:** run realineado (2026-07-09).
- **Entrada:** `stage02_xcorr_cube_stack.fits` (native_stage02); **modelo:** plano con sigma-clip 3σ, mediana 410 px por ajuste.
- **Efecto:** elimina el gradiente del halo estelar en la vecindad del compañero → fondo local plano para la extracción.
- **Máscaras:** núcleo del compañero 3 px, otros objetos 3 px; láser AO (5780–6050 Å, 216 canales) marcado como malo.
- **Rol:** alimenta la rama de extracción local-surface (`optimal_ls`) y la cadena de objetos lejanos (06/07/08); no es el método canónico (psffit lo es).
